# Ingeniería de Características y Partición Estratificada

Una vez finalizada la limpieza y unificación estructural en la fase de análisis exploratorio (EDA), se procede a la ingeniería de características (Feature Engineering). Esta fase adecúa las variables para su procesamiento matemático por los algoritmos de clasificación.

El objetivo analítico consiste en la predicción simultánea de la categorización integral de un ticket. Para ello, se formula una variable objetivo compuesta y se define la variable predictora principal.

Las transformaciones implementadas son:
* **Construcción de la Variable Objetivo:** Concatenación de los atributos categóricos `queue`, `type` y `priority` en una única dimensión de clasificación.
* **Construcción de la Variable Predictora:** Agrupación semántica del asunto (`subject`) y el cuerpo del mensaje (`body`) para proporcionar el contexto textual completo.
* **Partición de Datos:** División de la muestra en conjuntos de entrenamiento (80%) y test (20%). Se aplica un método de estratificación para garantizar que las clases con baja representación estadística mantengan su proporción original en ambos subconjuntos.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import os

# 1. Cargamos el dataset limpio del EDA
df = pd.read_parquet("../data/processed/df_final_silver.parquet")
print(f"Total de tickets provenientes del EDA: {len(df)}")

# Nos quedamos de momento solo con los tickets en inglés para el entrenamiento base
df_en = df[df['language'] == 'en'].copy()
print(f"Tickets en inglés disponibles: {len(df_en)}")

# 2. Ingeniería de la variable objetivo (Y)
# Juntamos las 3 variables del CRM para crear la etiqueta final a predecir
df_en['target'] = df_en['queue'] + " - " + df_en['type'] + " - " + df_en['priority']
num_clases = df_en['target'].nunique()
print(f"Número de combinaciones (clases) generadas: {num_clases}")

# 3. Ingeniería de la variable predictora (X)
# Juntamos el asunto y el cuerpo en un solo texto
df_en['texto_completo'] = df_en['subject'] + " " + df_en['body']

# Filtramos para quedarnos solo con las dos columnas que irán al modelo
df_ml = df_en[['texto_completo', 'target']].dropna()

# 4. Partición Train / Test
X = df_ml['texto_completo']
y = df_ml['target']

# Es vital usar stratify=y para no romper la distribución de las 84 clases
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nReparto de datos:")
print(f"Train: {len(X_train)} tickets")
print(f"Test: {len(X_test)} tickets")

# 5. Guardado físico
ruta_features = "../data/features/"
os.makedirs(ruta_features, exist_ok=True)

# Guardamos los textos y los targets en archivos separados
X_train.to_csv(ruta_features + "en_X_train_text.csv", index=False)
X_test.to_csv(ruta_features + "en_X_test_text.csv", index=False)
y_train.to_csv(ruta_features + "en_y_train.csv", index=False)
y_test.to_csv(ruta_features + "en_y_test.csv", index=False)

print("\nArchivos guardados correctamente en data/features/")

Total de tickets provenientes del EDA: 23867
Tickets en inglés disponibles: 23117
Número de combinaciones (clases) generadas: 84

Reparto de datos:
Train: 18493 tickets
Test: 4624 tickets

Archivos guardados correctamente en data/features/
